# TerraFoma - Integrity Score Model Training

This notebook trains a machine learning model to predict the **Integrity Score** (0-100) for carbon credits.
The Integrity Score gives buyers confidence in credit quality by evaluating:

- **Data Quality** (30%): Satellite imagery reliability
- **Model Confidence** (20%): Biomass estimation accuracy
- **Temporal Stability** (20%): Land cover consistency over time
- **Risk Factors** (15%): Environmental and political risks
- **Additionality** (15%): Conservation impact vs. baseline

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.metrics import mean_absolute_error, r2_score, mean_squared_error
from sklearn.inspection import permutation_importance, PartialDependenceDisplay
import joblib
import warnings
warnings.filterwarnings('ignore')

np.random.seed(42)
sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (12, 6)

## 1. Synthetic Data Generation

We generate 500 samples with 12 features that represent real-world factors affecting carbon credit integrity.

In [ ]:
N = 500

# Feature generation
ndvi_mean = np.random.uniform(0.15, 0.90, N)
ndvi_std = np.random.uniform(0.01, 0.25, N)
temporal_ndvi_change = np.random.uniform(-0.25, 0.25, N)
cloud_cover_pct = np.random.uniform(0, 80, N)
scan_resolution_m = np.random.choice([10, 20, 30, 60], N, p=[0.4, 0.3, 0.2, 0.1])
biomass_model_r2 = np.random.uniform(0.55, 0.98, N)
drought_risk = np.random.uniform(0, 0.9, N)
wildfire_risk = np.random.uniform(0, 0.8, N)
deforestation_proximity_km = np.random.uniform(0.5, 50, N)
years_under_conservation = np.random.uniform(0, 20, N)
land_use_encoded = np.random.choice([0, 1, 2, 3, 4], N, p=[0.35, 0.25, 0.2, 0.15, 0.05])
additionality_score = np.random.uniform(0.1, 0.95, N)

# Target: integrity_score (0-100)
# Data quality component (30%)
data_quality = (
    np.clip(ndvi_mean / 0.85, 0, 1) * 12 +
    np.clip(1 - ndvi_std / 0.20, 0, 1) * 6 +
    np.clip(1 - cloud_cover_pct / 80, 0, 1) * 6 +
    np.clip(1 - scan_resolution_m / 60, 0, 1) * 6
)

# Model confidence component (20%)
model_conf = biomass_model_r2 * 20

# Temporal stability component (20%)
temporal = (
    np.clip(1 - np.abs(temporal_ndvi_change) / 0.25, 0, 1) * 10 +
    np.clip(years_under_conservation / 15, 0, 1) * 10
)

# Risk component (15%)
risk_comp = (
    (1 - drought_risk) * 5 +
    (1 - wildfire_risk) * 5 +
    np.clip(deforestation_proximity_km / 30, 0, 1) * 5
)

# Additionality component (15%)
additionality_comp = additionality_score * 15

# Combine + noise
integrity_score = data_quality + model_conf + temporal + risk_comp + additionality_comp
integrity_score += np.random.normal(0, 3, N)  # noise
integrity_score = np.clip(integrity_score, 0, 100)

# Create DataFrame
df = pd.DataFrame({
    'ndvi_mean': ndvi_mean,
    'ndvi_std': ndvi_std,
    'temporal_ndvi_change': temporal_ndvi_change,
    'cloud_cover_pct': cloud_cover_pct,
    'scan_resolution_m': scan_resolution_m,
    'biomass_model_r2': biomass_model_r2,
    'drought_risk': drought_risk,
    'wildfire_risk': wildfire_risk,
    'deforestation_proximity_km': deforestation_proximity_km,
    'years_under_conservation': years_under_conservation,
    'land_use_encoded': land_use_encoded,
    'additionality_score': additionality_score,
    'integrity_score': integrity_score,
})

print(f'Dataset shape: {df.shape}')
print(f'\nIntegrity Score stats:')
print(df['integrity_score'].describe())

## 2. Exploratory Data Analysis

In [ ]:
# Feature distributions
fig, axes = plt.subplots(3, 4, figsize=(16, 10))
features = [c for c in df.columns if c != 'integrity_score']
for i, feat in enumerate(features):
    ax = axes[i // 4, i % 4]
    ax.hist(df[feat], bins=30, color='#16a34a', alpha=0.7, edgecolor='white')
    ax.set_title(feat, fontsize=10)
    ax.set_ylabel('Count')
plt.suptitle('Feature Distributions', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# Integrity score distribution
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].hist(df['integrity_score'], bins=40, color='#16a34a', alpha=0.8, edgecolor='white')
axes[0].set_xlabel('Integrity Score')
axes[0].set_ylabel('Count')
axes[0].set_title('Distribution of Integrity Scores')
axes[0].axvline(df['integrity_score'].mean(), color='red', linestyle='--', label=f'Mean: {df["integrity_score"].mean():.1f}')
axes[0].legend()

# By land use type
land_use_names = {0: 'Forest', 1: 'Agroforestry', 2: 'Grassland', 3: 'Cropland', 4: 'Wetland'}
for lu, name in land_use_names.items():
    subset = df[df['land_use_encoded'] == lu]['integrity_score']
    axes[1].hist(subset, bins=25, alpha=0.5, label=name)
axes[1].set_xlabel('Integrity Score')
axes[1].set_ylabel('Count')
axes[1].set_title('Integrity Score by Land Use Type')
axes[1].legend()

plt.tight_layout()
plt.show()

In [ ]:
# Correlation heatmap
plt.figure(figsize=(12, 8))
corr = df.corr()
mask = np.triu(np.ones_like(corr, dtype=bool))
sns.heatmap(corr, mask=mask, annot=True, fmt='.2f', cmap='RdYlGn',
            center=0, square=True, linewidths=0.5)
plt.title('Feature Correlation Heatmap', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# Key scatter plots
fig, axes = plt.subplots(2, 3, figsize=(16, 10))
key_features = ['ndvi_mean', 'biomass_model_r2', 'additionality_score',
                'drought_risk', 'cloud_cover_pct', 'years_under_conservation']
for i, feat in enumerate(key_features):
    ax = axes[i // 3, i % 3]
    ax.scatter(df[feat], df['integrity_score'], alpha=0.4, s=15, color='#16a34a')
    ax.set_xlabel(feat)
    ax.set_ylabel('Integrity Score')
    ax.set_title(f'{feat} vs Integrity Score')
plt.suptitle('Key Features vs Integrity Score', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

## 3. Model Training & Comparison

In [ ]:
# Prepare data
feature_cols = [c for c in df.columns if c != 'integrity_score']
X = df[feature_cols]
y = df['integrity_score']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
print(f'Train set: {X_train.shape[0]} samples')
print(f'Test set:  {X_test.shape[0]} samples')

In [ ]:
# Train 3 models
models = {
    'Linear Regression': LinearRegression(),
    'Random Forest': RandomForestRegressor(n_estimators=100, max_depth=12, random_state=42),
    'Gradient Boosting': GradientBoostingRegressor(n_estimators=150, max_depth=5, learning_rate=0.1, random_state=42),
}

results = {}
for name, model in models.items():
    model.fit(X_train, y_train)
    preds = model.predict(X_test)
    r2 = r2_score(y_test, preds)
    mae = mean_absolute_error(y_test, preds)
    rmse = np.sqrt(mean_squared_error(y_test, preds))
    results[name] = {'R2': r2, 'MAE': mae, 'RMSE': rmse, 'model': model}
    print(f'{name:25s} | R2: {r2:.4f} | MAE: {mae:.2f} | RMSE: {rmse:.2f}')

In [ ]:
# Model comparison bar chart
fig, axes = plt.subplots(1, 3, figsize=(15, 5))
model_names = list(results.keys())
colors = ['#94a3b8', '#16a34a', '#0891b2']

for i, metric in enumerate(['R2', 'MAE', 'RMSE']):
    values = [results[n][metric] for n in model_names]
    bars = axes[i].bar(model_names, values, color=colors)
    axes[i].set_title(metric, fontweight='bold')
    axes[i].set_ylabel(metric)
    for bar, val in zip(bars, values):
        axes[i].text(bar.get_x() + bar.get_width()/2, bar.get_height(),
                    f'{val:.3f}', ha='center', va='bottom', fontsize=10)

plt.suptitle('Model Comparison', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

## 4. Hyperparameter Tuning (Best Model)

In [ ]:
# Select best model and tune
best_name = max(results, key=lambda k: results[k]['R2'])
print(f'Best model: {best_name}')

# GridSearch on Gradient Boosting (typically best)
param_grid = {
    'n_estimators': [100, 150, 200],
    'max_depth': [4, 5, 6],
    'learning_rate': [0.05, 0.1, 0.15],
}

gb = GradientBoostingRegressor(random_state=42)
grid_search = GridSearchCV(gb, param_grid, cv=5, scoring='r2', n_jobs=-1, verbose=0)
grid_search.fit(X_train, y_train)

best_model = grid_search.best_estimator_
best_preds = best_model.predict(X_test)
print(f'\nBest params: {grid_search.best_params_}')
print(f'Best CV R2:  {grid_search.best_score_:.4f}')
print(f'Test R2:     {r2_score(y_test, best_preds):.4f}')
print(f'Test MAE:    {mean_absolute_error(y_test, best_preds):.2f}')
print(f'Test RMSE:   {np.sqrt(mean_squared_error(y_test, best_preds)):.2f}')

In [ ]:
# Predicted vs Actual scatter plot
fig, ax = plt.subplots(figsize=(8, 8))
ax.scatter(y_test, best_preds, alpha=0.5, s=20, color='#16a34a')
ax.plot([0, 100], [0, 100], 'r--', linewidth=2, label='Perfect prediction')
ax.set_xlabel('Actual Integrity Score', fontsize=12)
ax.set_ylabel('Predicted Integrity Score', fontsize=12)
ax.set_title('Predicted vs Actual Integrity Score', fontsize=14, fontweight='bold')
ax.legend()
ax.set_xlim(0, 100)
ax.set_ylim(0, 100)
plt.tight_layout()
plt.show()

## 5. Model Interpretation

In [ ]:
# Feature importance
importances = best_model.feature_importances_
feat_imp = pd.Series(importances, index=feature_cols).sort_values(ascending=True)

fig, ax = plt.subplots(figsize=(10, 7))
feat_imp.plot(kind='barh', color='#16a34a', ax=ax)
ax.set_xlabel('Feature Importance')
ax.set_title('Feature Importance (Gradient Boosting)', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

print('\nFeature importances:')
for feat, imp in feat_imp.sort_values(ascending=False).items():
    print(f'  {feat:35s} {imp:.4f}')

In [ ]:
# Permutation importance (more reliable)
perm_imp = permutation_importance(best_model, X_test, y_test, n_repeats=10, random_state=42)
perm_imp_df = pd.Series(perm_imp.importances_mean, index=feature_cols).sort_values(ascending=True)

fig, ax = plt.subplots(figsize=(10, 7))
perm_imp_df.plot(kind='barh', color='#0891b2', ax=ax)
ax.set_xlabel('Permutation Importance (decrease in R2)')
ax.set_title('Permutation Feature Importance', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# Partial dependence plots for top 3 features
top_features = feat_imp.sort_values(ascending=False).head(3).index.tolist()
print(f'Top 3 features: {top_features}')

fig, axes = plt.subplots(1, 3, figsize=(16, 5))
for i, feat in enumerate(top_features):
    PartialDependenceDisplay.from_estimator(
        best_model, X_train, [feat], ax=axes[i],
        line_kw={'color': '#16a34a', 'linewidth': 2}
    )
    axes[i].set_title(f'Partial Dependence: {feat}', fontsize=11)
plt.suptitle('How Top Features Affect Integrity Score', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

## 6. Export Model

In [ ]:
import os

# Save model
model_path = os.path.join('..', 'backend', 'ml', 'integrity_model.pkl')
os.makedirs(os.path.dirname(model_path), exist_ok=True)
joblib.dump(best_model, model_path)
print(f'Model saved to: {model_path}')

# Save feature names for inference
feature_info = {
    'feature_names': feature_cols,
    'model_type': type(best_model).__name__,
    'best_params': grid_search.best_params_,
    'test_r2': r2_score(y_test, best_preds),
    'test_mae': mean_absolute_error(y_test, best_preds),
}
joblib.dump(feature_info, os.path.join('..', 'backend', 'ml', 'integrity_model_info.pkl'))
print(f'Feature info saved.')
print(f'\nFeature names (in order): {feature_cols}')

In [ ]:
# Sample predictions on demo plots
demo_plots = pd.DataFrame([
    {'ndvi_mean': 0.78, 'ndvi_std': 0.05, 'temporal_ndvi_change': 0.03,
     'cloud_cover_pct': 10, 'scan_resolution_m': 10, 'biomass_model_r2': 0.92,
     'drought_risk': 0.15, 'wildfire_risk': 0.1, 'deforestation_proximity_km': 25,
     'years_under_conservation': 12, 'land_use_encoded': 0, 'additionality_score': 0.85},
    {'ndvi_mean': 0.45, 'ndvi_std': 0.18, 'temporal_ndvi_change': -0.12,
     'cloud_cover_pct': 45, 'scan_resolution_m': 30, 'biomass_model_r2': 0.72,
     'drought_risk': 0.55, 'wildfire_risk': 0.4, 'deforestation_proximity_km': 5,
     'years_under_conservation': 2, 'land_use_encoded': 2, 'additionality_score': 0.35},
    {'ndvi_mean': 0.62, 'ndvi_std': 0.10, 'temporal_ndvi_change': 0.08,
     'cloud_cover_pct': 20, 'scan_resolution_m': 10, 'biomass_model_r2': 0.88,
     'drought_risk': 0.25, 'wildfire_risk': 0.15, 'deforestation_proximity_km': 18,
     'years_under_conservation': 7, 'land_use_encoded': 1, 'additionality_score': 0.70},
])

demo_preds = best_model.predict(demo_plots)
labels = ['High-quality forest', 'Risky grassland', 'Medium agroforestry']
print('Sample Integrity Score Predictions:')
print('-' * 50)
for label, score in zip(labels, demo_preds):
    score = np.clip(score, 0, 100)
    quality = 'HIGH' if score >= 80 else 'MEDIUM' if score >= 60 else 'LOW'
    print(f'  {label:25s} -> {score:.1f}/100 ({quality})')